In [0]:
from datetime import date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
hist = [
    ("Alice", 34, "HR"),
    ("Bob", 45, "Finance"),
    ("Cathy", 29, "IT")
]
schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True)
])

df_hist = spark.createDataFrame(hist,schema)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS SCD2;
CREATE SCHEMA IF NOT EXISTS SCD2.BR;
CREATE SCHEMA IF NOT EXISTS SCD2.SIL;

In [0]:
df_inc.write.format('Delta').mode('append').saveAsTable('SCD2.BR.tb')

In [0]:
%sql
Select * from scd2.BR.tb

In [0]:
inc_data=[
    ("Alice", 37, "HR"),
    ("britney",67,"IT")]

schema2 = StructType([
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True)])

df_inc = spark.createDataFrame(inc_data, schema2)
df_inc.show()

In [0]:
from datetime import date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
sil = [
    ("Alice", 34, "HR",1,date(2002,1,1),date(9999,1,1)),
    ("Bob", 45, "Finance",1,date(2002,1,1),date(9999,1,1)),
    ("Cathy", 29, "IT",1,date(2002,1,1),date(9999,1,1))
]
schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True),
    StructField("is_active", IntegerType(), True),
    StructField("Start_date",DateType(), True),   
    StructField("End_date", DateType(), True)
])
df_sil = spark.createDataFrame(sil,schema)

In [0]:
df_sil.write.format('Delta').saveAsTable('SCD2.SIL.sil')

In [0]:
join=df_inc.join(df_sil,(df_inc.Name == df_sil.Name) & (df_sil.is_active ==1),'left_outer').select(df_inc['*'],\
    df_sil.Name.alias('sil_name'),df_sil.Age.alias('sil_age'),df_sil.Department.alias('sil_department'),df_sil.is_active.alias('sil_is_active'),df_sil.Start_date.alias('sil_start_date'),df_sil.End_date.alias('sil_end_date'))

In [0]:
from pyspark.sql.functions import *
f_df=join.filter(xxhash64(join.Name,join.Age,join.Department)!=xxhash64(join.sil_name,join.sil_age,join.sil_department))

In [0]:
%sql
MERGE INTO scd2.sil.sil AS target
USING SCD2.BR.tb AS source
ON target.Name = source.Name AND target.is_active = 1
WHEN MATCHED AND (target.Age <> source.Age OR target.Department <> source.Department)
  THEN UPDATE SET target.End_date = current_date(), target.is_active = 0
WHEN NOT MATCHED
  THEN INSERT (Name, Age, Department, Start_date, End_date, is_active)
       VALUES (source.Name, source.Age, source.Department, current_date(), NULL, 1);


In [0]:
%sql
Select * from scd2.sil.sil